# Why do $J_{ij}$'s not learn if $T$ is large?

I observed that just altering $T=3$ to $T=8$ drastically worsened the accuracy (rather, the model didn't learn well). 

This notebook tries to understand why.

Good Model details:
- Architecture: 3-layer AKOrN with channels [128, 256, 512]
- Oscillator dimension: n=2 (complex oscillators)
- Time steps: T=3 per layer
- No bias in convolution
- Best accuracy: 72% on CIFAR-10


In [ ]:
import sys
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import json
from pathlib import Path
import einops
from einops import rearrange
from sklearn.decomposition import PCA
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Add source directory to path
#sys.path.append('/source')
from source.models.classification.my_knet import MyAKOrN
from source.data.augs import augmentation_strong

# Set style for better plots
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Load Learned Model and Configuration

$T = 3, ~ \gamma = 1.0$

In [ ]:
# Load the best model checkpoint
checkpoint_path = "results/20250704_570979.opbs/my_akorn_cifar10_final.pth"
config_path = "results/20250704_570979.opbs/parameters.json"

# Load configuration
with open(config_path, 'r') as f:
    config = json.load(f)

print("Model Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

# Load checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device)
if 'epoch' in checkpoint_path:
    print(f"\nLoaded checkpoint from epoch {checkpoint['epoch']} with loss {checkpoint['loss']:.4f}")
elif 'final' in checkpoint_path:
    print(f"\nLoaded final checkpoint with accuracy {checkpoint['final_accuracy']:.2f}%")

# Create model with same configuration
model =MyAKOrN(
    n=config['n'],
    ch=config['ch'], 
    out_classes=config['num_classes'],
    L=config['L'],
    T=config['T'],
    J=config['J'],
    J_bias=config['J_bias'],
    ksizes=config['ksizes'],
    ro_ksize=config['ro_ksize'],
    ro_N=config['ro_N'],
    norm=config['norm'],
    c_norm=config['c_norm'],
    gamma=config['gamma'],
    use_omega=config['use_omega'],
    init_omg=config['init_omg'],
    global_omg=config['global_omg'],
    learn_omg=config['learn_omg'],
    ensemble=config['ensemble']
).to(device)

# Load state dict
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f"\nModel loaded successfully!")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
checkpoint['model_state_dict']

## 1. Create model with same parameters as in the good model but with $T=8$ 

In [ ]:
# Create model with same configuration
model_T8 =MyAKOrN(
    n=config['n'],
    ch=config['ch'], 
    out_classes=config['num_classes'],
    L=config['L'],
    T=8, #config['T'],
    J=config['J'],
    J_bias=config['J_bias'],
    ksizes=config['ksizes'],
    ro_ksize=config['ro_ksize'],
    ro_N=config['ro_N'],
    norm=config['norm'],
    c_norm=config['c_norm'],
    gamma=config['gamma'],
    use_omega=config['use_omega'],
    init_omg=config['init_omg'],
    global_omg=config['global_omg'],
    learn_omg=config['learn_omg'],
    ensemble=config['ensemble']
).to(device)

# Load state dict
model_T8.load_state_dict(checkpoint['model_state_dict'])
model_T8.eval()

In [ ]:
# CIFAR10のテストデータローダーを作成
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=config['batch_size'], shuffle=False, num_workers=config['num_workers'])

# accuracy計算関数
def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            preds = logits.argmax(dim=1)
            correct += (preds == y).sum().item()
            total += y.size(0)
    return correct / total


In [ ]:
# Needs to activate GPU to run this cell (or computation takes forever)
# Good model, shoud be about 72%
test_acc_gd = evaluate(model, test_loader, device)
# T=8
test_acc_8 = evaluate(model_T8, test_loader, device)

print(f"Test accuracy (T={config['T']}): {test_acc_gd*100:.2f}%")
print(f"Test accuracy (T=8): {test_acc_8*100:.2f}%")

## Check energy dynamics

In [ ]:
# Load CIFAR-10 for testing dynamics
transform_test = transforms.Compose([
    transforms.ToTensor(),
])

test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# CIFAR-10 class names
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

# Get a sample image
idx_img = 17
sample_image, sample_label = test_dataset[idx_img]
sample_image = sample_image.unsqueeze(0).to(device)
# sample_image, sample_label = next(iter(test_loader))
# sample_image = sample_image.to(device)

print(f"Sample image shape: {sample_image.shape}")
print(f"Sample label: {sample_label} ({classes[sample_label]})")
#print(f"Sample label: {sample_label.item()} ({classes[sample_label.item()]})")

# Visualize the sample
img_np = sample_image[0].cpu().permute(1, 2, 0).numpy()
plt.figure(figsize=(6, 6))
plt.imshow(img_np)
plt.title(f'Sample CIFAR-10 Image: {classes[sample_label]}')
#plt.title(f'Sample CIFAR-10 Image: {classes[sample_label.item()]}')
plt.axis('off')
plt.show()


In [ ]:
img_tensor = torch.from_numpy(img_np).permute(2, 0, 1).unsqueeze(0).float().to(device)
print(f"img_tensor shape: {img_tensor.shape}")

In [ ]:
torch.from_numpy(img_np).shape

In [ ]:
def depict_energy_dynamics(model, img_np, device=device):

    _, _, xs, es = model.feature(torch.from_numpy(img_np).permute(2, 0, 1).unsqueeze(0).float().to(device))

    plt.figure(figsize=(8, 5))
    for i, e in enumerate(es):
        # e is a list of tensors, convert to float for plotting
        e_np = [float(val.item()) for val in e]
        plt.plot(e_np, marker='o', label=f'Layer {i+1}')
    plt.xlabel('Time step')
    plt.ylabel('Energy')
    plt.title('Energy dynamics per layer')
    plt.legend()
    plt.grid(True)
    plt.show()

depict_energy_dynamics(model, img_np)

In [ ]:
depict_energy_dynamics(model_T8, img_np)

### Energy in another model


In [ ]:
# Load the best model checkpoint
checkpoint_path = "results/20250708_580356.opbs/my_akorn_cifar10_final.pth"
config_path = "results/20250708_580356.opbs/parameters.json"

# Load configuration
with open(config_path, 'r') as f:
    config = json.load(f)

print("Model Configuration:")
for key, value in config.items():
    print(f"  {key}: {value}")

# Load checkpoint
checkpoint = torch.load(checkpoint_path, map_location=device)
if 'epoch' in checkpoint_path:
    print(f"\nLoaded checkpoint from epoch {checkpoint['epoch']} with loss {checkpoint['loss']:.4f}")
elif 'final' in checkpoint_path:
    print(f"\nLoaded final checkpoint with accuracy {checkpoint['final_accuracy']:.2f}%")

# Create model with same configuration
model_gmin1_T8 =MyAKOrN(
    n=config['n'],
    ch=config['ch'], 
    out_classes=config['num_classes'],
    L=config['L'],
    T=8,#config['T'],
    J=config['J'],
    J_bias=config['J_bias'],
    ksizes=config['ksizes'],
    ro_ksize=config['ro_ksize'],
    ro_N=config['ro_N'],
    norm=config['norm'],
    c_norm=config['c_norm'],
    gamma=config['gamma'],
    use_omega=config['use_omega'],
    init_omg=config['init_omg'],
    global_omg=config['global_omg'],
    learn_omg=config['learn_omg'],
    ensemble=config['ensemble']
).to(device)

# Load state dict
model_gmin1_T8.load_state_dict(checkpoint['model_state_dict'])
model_gmin1_T8.eval()

print(f"\nModel loaded successfully!")
print(f"Total parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
depict_energy_dynamics(model_gmin1_T8, img_np)

In [ ]:
# Create model with same configuration
model_gmin1_T8_cfT32 = MyAKOrN(
    n=config['n'],
    ch=config['ch'], 
    out_classes=config['num_classes'],
    L=config['L'],
    T=32,#config['T'],
    J=config['J'],
    J_bias=config['J_bias'],
    ksizes=config['ksizes'],
    ro_ksize=config['ro_ksize'],
    ro_N=config['ro_N'],
    norm=config['norm'],
    c_norm=config['c_norm'],
    gamma=config['gamma'],
    use_omega=config['use_omega'],
    init_omg=config['init_omg'],
    global_omg=config['global_omg'],
    learn_omg=config['learn_omg'],
    ensemble=config['ensemble']
).to(device)


# Load state dict
model_gmin1_T8_cfT32.load_state_dict(checkpoint['model_state_dict'])
model_gmin1_T8_cfT32.eval()

In [ ]:
depict_energy_dynamics(model_gmin1_T8_cfT32, img_np)

In [ ]:
# Needs to activate GPU to run this cell (or computation takes forever)

test_acc_gmin1_T8 = evaluate(model_gmin1_T8, test_loader, device)
# T=8
test_acc_gmin1_T8_cfT32 = evaluate(model_gmin1_T8_cfT32, test_loader, device)

print(f"Test accuracy (gamma=0.1, T={config['T']}): {test_acc_gmin1_T8*100:.2f}%")
print(f"Test accuracy (gamma=0,1, T=32 but using learned params of T={config['T']}): {test_acc_gmin1_T8_cfT32*100:.2f}%")